# 🧪 Semana 8 · Unidad 3 — Laboratorio: Insertion Sort, Shell Sort y más

**Universidad de Talca — Curso de Algoritmos y Estructuras de Datos**

| Aspecto | Detalle |
|--------|--------|
| **Profesor** | PhD. César Astudillo |
| **Unidad** | Unidad 3: Ordenamiento |
| **Tema** | Insertion Sort · Bubble Sort · Shell Sort · Counting Sort |
| **Duración** | 100 minutos (2 bloques de 50 min) |
| **Fecha** | 2026-05 |

---

## 📋 Instrucciones Generales

- Trabaja de forma **individual**.
- Ejecuta cada celda antes de pasar a la siguiente.
- Las pruebas automáticas usan `unittest`, la librería estándar de Python (ver S00 · Unit Testing): cada prueba termina en `ok`, `FAIL` (resultado incorrecto) o `ERROR` (tu código lanzó una excepción). Apunta a que todas las celdas de prueba cierren con `OK`.
- Al terminar un bloque, el ayudante revisará tu avance antes de continuar.
- **No modifiques** las celdas de Setup ni de verificación automática.

## Setup (NO MODIFICAR)

In [ ]:
# Setup del laboratorio — ejecutar primero
import sys, random, time, timeit
import unittest

print("✅ Imports correctos")
print("🐍 Python", sys.version.split()[0])

# ── Implementaciones de referencia (usadas en los benchmarks) ─────────────

def _insertion_ref(a):
    a = a[:]
    for i in range(1, len(a)):
        c = a[i]; j = i - 1
        while j >= 0 and a[j] > c: a[j+1] = a[j]; j -= 1
        a[j+1] = c
    return a

def _bubble_ref(a):
    a = a[:]
    for i in range(len(a)-1):
        hubo = False
        for j in range(len(a)-1-i):
            if a[j] > a[j+1]: a[j], a[j+1] = a[j+1], a[j]; hubo = True
        if not hubo: break
    return a

def _shell_ref(a, gaps=None):
    a = a[:]
    n = len(a)
    if gaps is None:
        gaps = [1]                      # la secuencia de Knuth parte en 1
        g = 1
        while g < n//3: g = 3*g+1; gaps.append(g)
        gaps = sorted(gaps, reverse=True)
    for gap in gaps:
        for i in range(gap, n):
            c = a[i]; j = i - gap
            while j >= 0 and a[j] > c: a[j+gap] = a[j]; j -= gap
            a[j+gap] = c
    return a

def _counting_ref(a, k=None):
    if not a: return []
    if k is None: k = max(a) + 1
    cnt = [0]*k
    for v in a: cnt[v] += 1
    res = []
    for v in range(k): res.extend([v]*cnt[v])
    return res

# ── Gráfico de barras en texto (usado en los benchmarks) ──────────────────

def mostrar_barras(filas, unidad="ms", ancho=40, formato="8.2f", maximo=None):
    """
    Imprime un gráfico de barras horizontal en texto.

    Parámetros:
        filas (list[tuple[str, float]]): pares (etiqueta, valor).
        unidad (str): unidad que se muestra junto a cada valor.
        ancho (int): caracteres de la barra que representa a `maximo`.
        formato (str): formato del valor impreso, por ejemplo "8.2f" o "8,.0f".
        maximo (float | None): valor que ocupa todo el ancho; por defecto, el mayor
            de `filas`. Pasar el mismo valor a dos gráficos los deja en la misma escala.
    """
    maximo = maximo or max(valor for _, valor in filas) or 1
    margen = max(len(etiqueta) for etiqueta, _ in filas)
    for etiqueta, valor in filas:
        barra = "█" * round(valor / maximo * ancho) or ("▏" if valor > 0 else "")
        print(f"{etiqueta:>{margen}} │{barra:<{ancho}} {valor:{formato}} {unidad}")


print("✅ Implementaciones de referencia cargadas")

---
# 🔵 BLOQUE 1 — Insertion Sort y Bubble Sort (50 minutos)

> Al terminar este bloque, levanta la mano para que el ayudante revise tu progreso.

## PARTE 1A: Trazar Insertion Sort a Mano (10 minutos)

Traza la ejecución de Insertion Sort **sin ejecutar código** sobre la lista:

**Lista:** `[7, 3, 9, 1, 5, 2, 8, 4]`

Completa la tabla indicando el estado del arreglo DESPUÉS de cada inserción  
y la posición final donde se insertó la clave:

| Paso i | Clave | Posición inserción | Estado del arreglo `a[0..i]` |
|--------|-------|-------------------|------------------------------|
| i=1    |       |                   |                              |
| i=2    |       |                   |                              |
| i=3    |       |                   |                              |
| i=4    |       |                   |                              |
| i=5    |       |                   |                              |
| i=6    |       |                   |                              |
| i=7    |       |                   |                              |

**Total desplazamientos:** ___  
**Inversiones originales:** ___ *(Recuerda: son lo mismo)*

### Tu Respuesta (edita esta celda)

_[Completa la tabla aquí]_

Desplazamientos totales: ___  
Inversiones: ___

In [ ]:
# Verificación de tu trazado
datos_trazado = [7, 3, 9, 1, 5, 2, 8, 4]

# Ejecuta para ver la solución y verificar tu tabla
a = datos_trazado[:]
desplazamientos = 0
for i in range(1, len(a)):
    clave = a[i]; j = i - 1
    while j >= 0 and a[j] > clave:
        a[j+1] = a[j]; j -= 1; desplazamientos += 1
    a[j+1] = clave
    print(f"  i={i}: clave={clave:2d} → insertada en pos {j+1}  | prefijo: {a[:i+1]}")

print(f"\nTotal desplazamientos: {desplazamientos}")

# Verificar inversiones
inversiones = sum(1 for i in range(len(datos_trazado))
                    for j in range(i+1, len(datos_trazado))
                    if datos_trazado[i] > datos_trazado[j])
print(f"Inversiones en lista original: {inversiones}\n")


class TestTrazado(unittest.TestCase):
    """Relación clave de Insertion Sort."""

    def test_desplazamientos_igual_inversiones(self):
        """el total de desplazamientos es igual al número de inversiones"""
        self.assertEqual(desplazamientos, inversiones)


resultado = unittest.main(argv=["ignorado", "TestTrazado"], exit=False, verbosity=2)

## PARTE 1B: Implementar Insertion Sort desde Cero (15 minutos)

Implementa `MiInsertionSort` como clase, con la misma API que `MiSelectionSort` del Lab 1.

In [ ]:
class MiInsertionSort:
    """
    Implementación propia de Insertion Sort.

    API:
        sort(lista) → lista ordenada
        comparaciones → número de comparaciones realizadas
        desplazamientos → número de desplazamientos realizados
    """

    def __init__(self):
        self.comparaciones    = 0
        self.desplazamientos  = 0

    def sort(self, lista: list) -> list:
        """
        Ordena lista y actualiza self.comparaciones y self.desplazamientos.

        Parámetros:
            lista (list): lista de elementos comparables
        Retorna:
            list: nueva lista ordenada
        """
        self.comparaciones   = 0
        self.desplazamientos = 0
        # Tu código aquí
        pass

In [ ]:
# Pruebas automáticas — MiInsertionSort (unittest)

class TestMiInsertionSort(unittest.TestCase):
    """MiInsertionSort ordena y cuenta desplazamientos (= inversiones)."""

    def ordenar(self, lista):
        inst = MiInsertionSort()
        return inst, inst.sort(lista[:])

    def test_1_ordena(self):
        """ordena listas general con repetidos, vacía, de un elemento y aleatoria"""
        casos = {
            "general con repetidos": [3, 1, 4, 1, 5, 9, 2, 6],
            "vacía": [],
            "un elemento": [1],
            "aleatoria n=20": random.sample(range(100), 20),
        }
        for descripcion, lista in casos.items():
            with self.subTest(descripcion):
                _, resultado = self.ordenar(lista)
                self.assertEqual(resultado, sorted(lista))

    def test_2_mejor_caso(self):
        """lista ya ordenada o con todos iguales: 0 desplazamientos"""
        for lista in ([1, 2, 3, 4, 5], [2, 2, 2, 2]):
            with self.subTest(lista=lista):
                inst, resultado = self.ordenar(lista)
                self.assertEqual(resultado, sorted(lista))
                self.assertEqual(inst.desplazamientos, 0)

    def test_3_peor_caso(self):
        """lista invertida de 5 elementos: n(n-1)/2 = 10 desplazamientos"""
        inst, resultado = self.ordenar([5, 4, 3, 2, 1])
        self.assertEqual(resultado, [1, 2, 3, 4, 5])
        self.assertEqual(inst.desplazamientos, 10)

    def test_4_desplazamientos_igual_inversiones(self):
        """desplazamientos coincide con el número de inversiones de la entrada"""
        lista = [3, 1, 4, 1, 5, 9, 2, 6]
        inversiones = sum(1 for i in range(len(lista)) for j in range(i + 1, len(lista)) if lista[i] > lista[j])
        inst, _ = self.ordenar(lista)
        self.assertEqual(inst.desplazamientos, inversiones)


resultado = unittest.main(argv=["ignorado", "TestMiInsertionSort"], exit=False, verbosity=2)

## PARTE 1C: Bubble Sort con Early-Exit (15 minutos)

Implementa `MiBubbleSort` con la optimización del flag `hubo_swap`.

In [ ]:
class MiBubbleSort:
    """
    Bubble Sort con optimización early-exit.

    API equivalente a MiInsertionSort:
        sort(lista) → lista ordenada
        comparaciones → número total de comparaciones
        swaps         → número total de intercambios
        pasadas       → número de pasadas realizadas (debe ser < n si usa early-exit)
    """

    def __init__(self):
        self.comparaciones = 0
        self.swaps         = 0
        self.pasadas       = 0

    def sort(self, lista: list) -> list:
        """
        Ordena lista con Bubble Sort y early-exit.
        Actualiza self.comparaciones, self.swaps, self.pasadas.
        """
        self.comparaciones = 0
        self.swaps         = 0
        self.pasadas       = 0
        # Tu código aquí
        pass

In [ ]:
# Pruebas automáticas — MiBubbleSort (unittest)

class TestMiBubbleSort(unittest.TestCase):
    """MiBubbleSort ordena, cuenta swaps y termina antes con el flag hubo_swap."""

    def ordenar(self, lista):
        inst = MiBubbleSort()
        return inst, inst.sort(lista[:])

    def test_1_ordena(self):
        """ordena listas general, vacía, de un elemento e invertida"""
        casos = {
            "general": [5, 3, 8, 1, 9, 2],
            "vacía": [],
            "un elemento": [1],
            "invertida": [5, 4, 3, 2, 1],
        }
        for descripcion, lista in casos.items():
            with self.subTest(descripcion):
                _, resultado = self.ordenar(lista)
                self.assertEqual(resultado, sorted(lista))

    def test_2_early_exit(self):
        """lista ya ordenada: 0 swaps y a lo más 2 pasadas"""
        inst, resultado = self.ordenar([1, 2, 3, 4, 5])
        self.assertEqual(resultado, [1, 2, 3, 4, 5])
        self.assertEqual(inst.swaps, 0)
        self.assertLessEqual(inst.pasadas, 2)

    def test_3_casi_ordenada(self):
        """[2, 1, 3, 4, 5] queda ordenada con un solo swap"""
        inst, resultado = self.ordenar([2, 1, 3, 4, 5])
        self.assertEqual(resultado, [1, 2, 3, 4, 5])
        self.assertEqual(inst.swaps, 1)


resultado = unittest.main(argv=["ignorado", "TestMiBubbleSort"], exit=False, verbosity=2)

## PARTE 1D: Comparación Visual (10 minutos)

In [ ]:
# Comparación en texto: Insertion Sort vs Bubble Sort en distintos escenarios
import timeit, random

def bench(fn, datos, reps=100):
    return timeit.timeit(lambda: fn(datos[:]), number=reps) / reps * 1000

ns = [50, 100, 200, 400, 600]
escenarios = {
    'Aleatorio':     lambda n: random.sample(range(n*3), n),
    'Ya ordenado':   lambda n: list(range(n)),
    'Invertido':     lambda n: list(range(n, 0, -1)),
}

print("Insertion Sort vs Bubble Sort — tiempo por ejecución")
for nombre, gen in escenarios.items():
    filas = []
    for n in ns:
        datos = gen(n)
        filas += [(f"n={n} insertion", bench(_insertion_ref, datos)),
                  ("bubble", bench(_bubble_ref, datos))]
    print(f"\nEscenario: {nombre}\n")
    mostrar_barras(filas)

print("\nObservaciones:")
print("  • Mejor caso (ya ordenado): ¿cuál gana? ¿por qué?")
print("  • Peor caso (invertido):    ¿cuál hace más swaps?")

## 📝 Pregunta de Reflexión — Bloque 1

**Responde en la celda siguiente (en texto, no en código):**

1. ¿En qué escenario Bubble Sort puede superar a Insertion Sort? ¿Por qué?
2. ¿Por qué el número de inversiones = número de desplazamientos de Insertion Sort?

### Tu Respuesta

_[Escribe aquí tu respuesta]_

---
# 🟣 BLOQUE 2 — Shell Sort y Counting Sort (50 minutos)

> Comienza este bloque cuando el ayudante haya revisado el Bloque 1.

## PARTE 2A: Trazar Shell Sort a Mano (10 minutos)

**Lista:** `[9, 4, 7, 2, 8, 1, 6, 3]`  
**Secuencia de gaps:** `[4, 2, 1]`

Para cada fase de gap, muestra cómo queda el arreglo después de completar  
el Insertion Sort con ese gap:

| Gap | Sub-listas comparadas | Estado tras la fase |
|-----|----------------------|---------------------|
| 4   | a[0],a[4] / a[1],a[5] / a[2],a[6] / a[3],a[7] | |
| 2   | a[0],a[2],a[4],a[6] / a[1],a[3],a[5],a[7] | |
| 1   | toda la lista (Insertion Sort) | |

### Tu Respuesta

_[Completa la tabla]_

In [ ]:
# Verificación del trazado
lista_traza = [9, 4, 7, 2, 8, 1, 6, 3]
gaps_traza  = [4, 2, 1]

a = lista_traza[:]
print(f"Estado inicial:   {a}")
for gap in gaps_traza:
    for i in range(gap, len(a)):
        c = a[i]; j = i - gap
        while j >= 0 and a[j] > c: a[j+gap] = a[j]; j -= gap
        a[j+gap] = c
    print(f"Tras gap={gap}:   {a}")

## PARTE 2B: Implementar MiShellSort (20 minutos)

Implementa Shell Sort con soporte para distintas secuencias de gaps.

In [ ]:
class MiShellSort:
    """
    Implementación propia de Shell Sort.

    API:
        sort(lista, gaps=None) → lista ordenada
        comparaciones → número de comparaciones
        swaps         → número de swaps
        gaps_usados   → lista de gaps efectivamente usados
    """

    def __init__(self):
        self.comparaciones = 0
        self.swaps         = 0
        self.gaps_usados   = []

    def _generar_gaps_knuth(self, n: int) -> list:
        """Genera la secuencia de Knuth: 1, 4, 13, 40, ... hasta < n/3"""
        # Tu código aquí
        pass

    def sort(self, lista: list, gaps: list = None) -> list:
        """
        Ordena lista con Shell Sort.
        Si gaps es None, usa la secuencia de Knuth.
        """
        self.comparaciones = 0
        self.swaps         = 0
        self.gaps_usados   = []
        # Tu código aquí
        pass

In [ ]:
# Pruebas automáticas — MiShellSort (unittest)

class TestMiShellSort(unittest.TestCase):
    """MiShellSort ordena con la secuencia de Knuth o con los gaps entregados."""

    def ordenar(self, lista, gaps=None):
        return MiShellSort().sort(lista[:], gaps[:] if gaps else None)

    def test_1_gaps_knuth(self):
        """sin gaps: ordena listas general, vacía, de un elemento, ordenada, invertida y aleatoria"""
        casos = {
            "general": [5, 2, 8, 1, 9, 3],
            "vacía": [],
            "un elemento": [7],
            "ya ordenada": [1, 2, 3, 4, 5],
            "invertida": [5, 4, 3, 2, 1],
            "aleatoria n=50": random.sample(range(200), 50),
        }
        for descripcion, lista in casos.items():
            with self.subTest(descripcion):
                self.assertEqual(self.ordenar(lista), sorted(lista))

    def test_2_gaps_explicitos(self):
        """con gaps entregados: todos iguales con [1] y n=20 invertida con [4, 2, 1]"""
        casos = [("todos iguales", [3, 3, 3], [1]),
                 ("n=20 invertida", list(range(20, 0, -1)), [4, 2, 1])]
        for descripcion, lista, gaps in casos:
            with self.subTest(descripcion):
                self.assertEqual(self.ordenar(lista, gaps), sorted(lista))


resultado = unittest.main(argv=["ignorado", "TestMiShellSort"], exit=False, verbosity=2)

## PARTE 2C: Benchmark — Shell Sort vs Insertion Sort (10 minutos)

In [ ]:
# Benchmark: Shell Sort vs Insertion Sort — curva de escalabilidad
import timeit, random

def insertion_simple(a):
    a = a[:]
    for i in range(1, len(a)):
        c = a[i]; j = i - 1
        while j >= 0 and a[j] > c: a[j+1] = a[j]; j -= 1
        a[j+1] = c
    return a

def shell_knuth(a):
    a = a[:]
    n = len(a)
    gaps = [1]                          # la secuencia de Knuth parte en 1
    g = 1
    while g < n//3: g = 3*g+1; gaps.append(g)
    gaps = sorted(gaps, reverse=True)
    for gap in gaps:
        for i in range(gap, n):
            c = a[i]; j = i - gap
            while j >= 0 and a[j] > c: a[j+gap] = a[j]; j -= gap
            a[j+gap] = c
    return a

ns = [100, 500, 1000, 2000, 5000, 10000]
t_ins, t_shell = [], []
for n in ns:
    datos = random.sample(range(n*3), n)
    t_ins.append(timeit.timeit(lambda: insertion_simple(datos), number=15) / 15 * 1000)
    t_shell.append(timeit.timeit(lambda: shell_knuth(datos),    number=15) / 15 * 1000)

speedup = [i/s for i, s in zip(t_ins, t_shell)]

print("Tiempo absoluto (ms por ejecución)\n")
filas = []
for n, ti, ts in zip(ns, t_ins, t_shell):
    filas += [(f"n={n} insertion", ti), ("shell (Knuth)", ts)]
mostrar_barras(filas)

print("\nSpeedup de Shell Sort sobre Insertion Sort\n")
mostrar_barras([(f"n={n}", sp) for n, sp in zip(ns, speedup)], unidad="×", formato="7.1f")

print(f"\n{'n':>6} | {'Insertion':>10} | {'Shell':>10} | {'Speedup':>8}")
print("-" * 40)
for n, ti, ts, sp in zip(ns, t_ins, t_shell, speedup):
    print(f"{n:>6} | {ti:>8.2f}ms | {ts:>8.2f}ms | {sp:>7.1f}×")

## PARTE 2D: Counting Sort — Desafío ⭐⭐ (10 minutos, opcional)

Implementa `counting_sort_estable(lista, k)` en la versión estable de 3 pasos  
(frecuencias → prefix sum → colocar de atrás hacia adelante).

In [ ]:
def counting_sort_estable(lista: list, k: int) -> list:
    """
    Counting Sort estable usando la técnica de prefix sum.
    
    Esta versión garantiza estabilidad: elementos iguales mantienen
    su orden relativo original.

    Pasos:
        1. Contar frecuencias en conteo[0..k-1]
        2. Calcular prefix sum: conteo[i] += conteo[i-1]
           (ahora conteo[v] = número de elementos ≤ v)
        3. Iterar de DERECHA A IZQUIERDA sobre lista original:
           colocar lista[i] en salida[conteo[lista[i]]-1] y decrementar conteo

    Parámetros:
        lista (list): lista de enteros en [0, k)
        k     (int):  rango exclusivo superior
    Retorna:
        list: lista ordenada de forma estable
    """
    # Tu código aquí
    pass

In [ ]:
# Pruebas automáticas — counting_sort_estable (unittest)

class TestCountingSortEstable(unittest.TestCase):
    """counting_sort_estable ordena enteros en el rango [0, k)."""

    CASOS = [
        ([4, 2, 2, 8, 3, 3, 1], 9, [1, 2, 2, 3, 3, 4, 8], "lista con repetidos"),
        ([],        5, [],           "lista vacía"),
        ([0],       1, [0],          "un elemento"),
        ([3, 2, 1, 0], 4, [0, 1, 2, 3], "invertida k=4"),
        ([0, 0, 0], 1, [0, 0, 0],    "todos iguales"),
    ]

    def test_casos(self):
        """resultado ordenado en los 5 casos"""
        for lista, k, esperado, descripcion in self.CASOS:
            with self.subTest(descripcion):
                self.assertEqual(counting_sort_estable(lista[:], k), esperado)


resultado = unittest.main(argv=["ignorado", "TestCountingSortEstable"], exit=False, verbosity=2)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN Counting Sort estable — ver solo si terminaste el tiempo
# ═══════════════════════════════════════════════════

# def counting_sort_estable(lista: list, k: int) -> list:
#     if not lista: return []
#     n = len(lista)
#     # Paso 1: frecuencias
#     conteo = [0] * k
#     for v in lista: conteo[v] += 1
#     # Paso 2: prefix sum (posiciones finales)
#     for i in range(1, k): conteo[i] += conteo[i-1]
#     # Paso 3: colocar de derecha a izquierda (estabilidad)
#     salida = [0] * n
#     for i in range(n-1, -1, -1):
#         v = lista[i]
#         conteo[v] -= 1
#         salida[conteo[v]] = v
#     return salida

## PARTE 2E: Tabla Comparativa Final — Los 5 Algoritmos (10 minutos)

In [ ]:
# Benchmark final: los 5 algoritmos lado a lado
import timeit, random

def counting_simple(a):
    if not a: return []
    k = max(a)+1; cnt=[0]*k
    for v in a: cnt[v]+=1
    res=[]
    for v in range(k): res.extend([v]*cnt[v])
    return res

def selection_sort(a):
    a = a[:]
    n = len(a)
    for i in range(n-1):
        idx = i
        for j in range(i+1, n):
            if a[j] < a[idx]: idx = j
        a[i], a[idx] = a[idx], a[i]
    return a

algoritmos = {
    'Selection': selection_sort,
    'Insertion': _insertion_ref,
    'Bubble':    _bubble_ref,
    'Shell':     _shell_ref,
    'Counting':  counting_simple,
}

ns = [100, 500, 1000, 3000]
filas = []
for n in ns:
    datos = random.sample(range(n*3), n)
    for k, (nombre, fn) in enumerate(algoritmos.items()):
        t = timeit.timeit(lambda: fn(datos[:]), number=20) / 20 * 1000
        filas.append((f"n={n} {nombre}" if k == 0 else nombre, t))

print("Los 5 algoritmos de la semana — tiempo por ejecución (los mismos datos para todos)\n")
mostrar_barras(filas)

## 📝 Pregunta de Reflexión — Bloque 2

**Responde en la celda siguiente:**

1. ¿Por qué Shell Sort supera a Insertion Sort incluso con datos aleatorios?
2. ¿En qué situación real usarías Counting Sort? Da un ejemplo concreto.
3. ¿Por qué la secuencia de gaps afecta tanto el rendimiento de Shell Sort?

### Tu Respuesta

_[Escribe aquí tu respuesta]_

---
## ✅ Checklist de Entrega

### Bloque 1 — Insertion Sort y Bubble Sort
- [ ] Tabla de trazado manual completada (PARTE 1A)
- [ ] `MiInsertionSort` implementada y `TestMiInsertionSort` en `OK`
- [ ] `MiBubbleSort` con early-exit implementada y `TestMiBubbleSort` en `OK`
- [ ] Comparación de tiempos generada (PARTE 1D)
- [ ] Pregunta de reflexión respondida

### Bloque 2 — Shell Sort y Counting Sort
- [ ] Tabla de trazado Shell Sort completada (PARTE 2A)
- [ ] `MiShellSort` implementada y `TestMiShellSort` en `OK`
- [ ] Benchmark ejecutado (PARTE 2C)
- [ ] Counting Sort estable intentada (PARTE 2D — opcional ⭐⭐)
- [ ] Tabla comparativa de los 5 algoritmos generada
- [ ] Preguntas de reflexión respondidas

---
**Nombre:** ___________________________  
**Historial:** v1.0 (2026-05) — Creación inicial